In [ ]:
import numpy as np
import pandas as pd

import pymc as pm
import arviz as az
import pytensor.tensor as pt

import matplotlib.pyplot as plt
import seaborn as sns

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

In [ ]:
df_latent = pd.read_csv("latent_motor_severity_score.csv")
df_clin = pd.read_excel("../data/raw/DatasetIcotMond.xlsx")

In [ ]:
# --- Latent severity artifact ---
df_latent = pd.read_csv("latent_motor_severity_score.csv")

print("Latent artifact shape:", df_latent.shape)
df_latent.head()

In [ ]:
df_clin = pd.read_excel("../data/raw/DatasetIcotMond.xlsx")

print("Clinical shape:", df_clin.shape)
df_clin.head()

In [ ]:
df_clin["ID"] = (
    df_clin["ID"]
    .astype(str)
    .str.strip()
    .str.split()
    .str[0]
    .astype(int)
)

In [ ]:
df_latent["ID"] = (
    df_latent["ID"]
    .astype(str)
    .str.strip()
    .str.split()
    .str[0]
    .astype(int)
)

In [ ]:
# =========================================
# FIX — Clean ID also in latent artifact
# =========================================

df_latent["ID"] = (
    df_latent["ID"]
    .astype(str)
    .str.strip()
    .str.split()
    .str[0]
    .astype(int)
)

print("Latent ID dtype:", df_latent["ID"].dtype)
print("Unique latent IDs:", df_latent["ID"].nunique())

In [ ]:
df_sev_subject = (
    df_latent
    .groupby("ID", as_index=False)
    .agg(
        latent_severity=("latent_severity", "median"),
        latent_severity_sd=("latent_severity_sd", "median"),
        n_obs=("latent_severity", "size")
    )
)

print("Aggregated severity shape:", df_sev_subject.shape)
df_sev_subject.head()

In [ ]:
df_clin_subject = (
    df_clin
    .groupby("ID", as_index=False)
    .agg(
        Age=("Age", "first"),
        Duration_years=("Duration (years)", "first"),
        target_bin=("target_bin", "max")
    )
)

In [ ]:
df_subject = df_clin_subject.merge(
    df_sev_subject,
    on="ID",
    how="inner",
    validate="one_to_one"
)

print("Subject-level dataset shape:", df_subject.shape)
print("Falls prevalence:", df_subject["target_bin"].mean().round(3))
df_subject.head()

In [ ]:
print("N subjects:", len(df_subject))
print("Min n_obs:", df_subject["n_obs"].min())
print("Max n_obs:", df_subject["n_obs"].max())
print("Subjects with single observation:", (df_subject["n_obs"] == 1).sum())

In [ ]:
import numpy as np
import pandas as pd

import pymc as pm
import arviz as az
import pytensor.tensor as pt

import matplotlib.pyplot as plt
import seaborn as sns

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

In [ ]:
df_subject = pd.read_csv("subject_level_latent_severity_and_falls.csv")

print("N subjects:", len(df_subject))
print("Fall rate:", df_subject["target_bin"].mean().round(3))

df_subject.head()

In [ ]:
from sklearn.preprocessing import StandardScaler

sc = StandardScaler()
df_subject["sev_z"] = sc.fit_transform(
    df_subject[["latent_severity"]]
).ravel()

df_subject["sev_sd"] = df_subject["latent_severity_sd"]

y = df_subject["target_bin"].astype(int).values

print("sev_z mean:", df_subject["sev_z"].mean().round(3))
print("sev_z sd:", df_subject["sev_z"].std().round(3))

In [ ]:
with pm.Model() as threshold_model:

    # -------------------------
    # THRESHOLD (τ)
    # -------------------------
    tau = pm.Normal(
        "tau",
        mu=0.0,
        sigma=1.0
    )

    # -------------------------
    # TRUE latent severity
    # (measurement error propagation)
    # -------------------------
    true_severity = pm.Normal(
        "true_severity",
        mu=df_subject["sev_z"].values,
        sigma=df_subject["sev_sd"].values,
        shape=len(df_subject)
    )

    # -------------------------
    # Probit threshold mechanism
    # -------------------------
    p_fall = pm.Deterministic(
        "p_fall",
        0.5 * (1 + pt.erf((true_severity - tau) / pt.sqrt(2)))
    )

    pm.Bernoulli(
        "falls",
        p=p_fall,
        observed=y
    )

    trace_threshold = pm.sample(
        draws=3000,
        tune=3000,
        chains=4,
        target_accept=0.95,
        random_seed=RANDOM_SEED
    )

In [ ]:
az.summary(
    trace_threshold,
    var_names=["tau"],
    hdi_prob=0.95
)

In [ ]:
# Posterior samples
post = trace_threshold.posterior

tau_samples = (
    post["tau"]
    .stack(sample=("chain", "draw"))
    .values
)

true_sev_samples = (
    post["true_severity"]
    .stack(sample=("chain", "draw"))
    .values
)

print("tau mean:", tau_samples.mean().round(3))
print("tau 95% HDI:",
      np.quantile(tau_samples, [0.025, 0.975]).round(3))

In [ ]:
from scipy.stats import norm

In [ ]:
# Severity grid (z-score)
sev_grid = np.linspace(-2.5, 2.5, 201)

# Probit = CDF normale standard
p_grid = norm.cdf(
    sev_grid[:, None] - tau_samples[None, :]
)

# Riassunti posterior
p_mean = p_grid.mean(axis=1)
p_lo = np.quantile(p_grid, 0.025, axis=1)
p_hi = np.quantile(p_grid, 0.975, axis=1)

In [ ]:
plt.figure(figsize=(7, 5))

# Curva media
plt.plot(sev_grid, p_mean, lw=2, label="P(fall | severity)")

# Banda credibile
plt.fill_between(sev_grid, p_lo, p_hi, alpha=0.25, label="95% credible interval")

# Soglia τ (media e HDI)
plt.axvline(tau_samples.mean(), color="red", ls="--", lw=2, label="Threshold τ (mean)")
plt.axvspan(
    np.quantile(tau_samples, 0.025),
    np.quantile(tau_samples, 0.975),
    color="red",
    alpha=0.12,
    label="τ 95% HDI"
)

# Aesthetics
plt.xlabel("Latent motor severity (z)")
plt.ylabel("P(fall)")
plt.title("Falls as threshold-crossing events of latent motor instability")
plt.ylim(0, 1)
plt.legend(frameon=False)
plt.tight_layout()
plt.show()

In [ ]:
# Definiamo zone operative
safe_zone = sev_grid < np.quantile(tau_samples, 0.025)
risk_zone = sev_grid > np.quantile(tau_samples, 0.975)

print("Safe-zone severity up to:", sev_grid[safe_zone][-1].round(2))
print("High-risk severity from:", sev_grid[risk_zone][0].round(2))

In [ ]:
from scipy.stats import norm
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
tau_samples = (
    trace_threshold.posterior["tau"]
    .stack(sample=("chain", "draw"))
    .values
)

len(tau_samples)

In [ ]:
sev_grid = np.linspace(-2.5, 2.5, 200)
delta = 0.3

In [ ]:
p_orig = norm.cdf(sev_grid[:, None] - tau_samples[None, :])
p_cf   = norm.cdf((sev_grid[:, None] - delta) - tau_samples[None, :])

In [ ]:
delta_p = p_cf - p_orig

delta_mean = delta_p.mean(axis=1)
delta_hdi_low = np.percentile(delta_p, 2.5, axis=1)
delta_hdi_high = np.percentile(delta_p, 97.5, axis=1)

# sanity check
delta_mean[:5]

In [ ]:
plt.figure(figsize=(8,5))

plt.plot(
    sev_grid,
    delta_mean,
    color="darkred",
    lw=2,
    label=r"$\Delta P(\mathrm{fall})$ for $\Delta s = -0.3$"
)

plt.fill_between(
    sev_grid,
    delta_hdi_low,
    delta_hdi_high,
    color="darkred",
    alpha=0.25,
    label="95% credible interval"
)

plt.axhline(0, color="black", ls="--", lw=1)

plt.xlabel("Latent motor severity (z)")
plt.ylabel("Change in fall probability")
plt.title("Counterfactual risk reduction from latent severity improvement")

plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ---- Sanity checks ----
assert "tau_samples" in globals(), "tau_samples non trovato. Esegui prima il modello threshold e salva i campioni di tau."
tau_samples = np.asarray(tau_samples).ravel()
assert tau_samples.ndim == 1 and len(tau_samples) > 1000, "tau_samples sembra troppo piccolo o non 1D."

# ---- Severity grid (se non esiste lo ricreo) ----
if "sev_grid" not in globals():
    sev_grid = np.linspace(-2.5, 2.5, 201)
else:
    sev_grid = np.asarray(sev_grid)

print("tau_samples:", tau_samples.shape, "| sev_grid:", sev_grid.shape)

In [ ]:
try:
    from scipy.special import ndtr  # Normal CDF
except Exception as e:
    raise ImportError(
        "Serve scipy per ndtr (Normal CDF). Installa con: pip install scipy"
    ) from e

def probit_cdf(x):
    return ndtr(x)   # Φ(x)

In [ ]:
def delta_posterior(sev_grid, tau_samples, delta):
    """
    Returns:
      delta_mean: mean ΔP over posterior
      delta_lo, delta_hi: 95% CrI over posterior
      delta_grid_samples: optional (grid x samples) if ti serve
    """
    # broadcasting: (G,1) - (1,S) -> (G,S)
    base = probit_cdf(sev_grid[:, None] - tau_samples[None, :])
    shifted = probit_cdf((sev_grid[:, None] - delta) - tau_samples[None, :])
    delta_grid = shifted - base  # ΔP(fall) for each (grid, posterior sample)

    delta_mean = delta_grid.mean(axis=1)
    delta_lo = np.quantile(delta_grid, 0.025, axis=1)
    delta_hi = np.quantile(delta_grid, 0.975, axis=1)

    return delta_mean, delta_lo, delta_hi

In [ ]:
deltas = [0.1, 0.2, 0.3, 0.5]

multi = {}
for d in deltas:
    dm, dlo, dhi = delta_posterior(sev_grid, tau_samples, d)
    multi[d] = {"mean": dm, "lo": dlo, "hi": dhi}

print("Computed multi-Δ curves for:", deltas)

In [ ]:
plt.figure(figsize=(9,6))

for d in deltas:
    dm = multi[d]["mean"]
    dlo = multi[d]["lo"]
    dhi = multi[d]["hi"]

    plt.plot(sev_grid, dm, lw=2, label=f"Δs = -{d}")
    plt.fill_between(sev_grid, dlo, dhi, alpha=0.15)

plt.axhline(0, ls="--", lw=1)
plt.xlabel("Latent motor severity (z)")
plt.ylabel("ΔP(fall) = P(fall | s-Δ) - P(fall | s)")
plt.title("Dose–response: counterfactual fall-risk reduction for severity improvements")
plt.legend()
plt.show()

In [ ]:
summary_rows = []

for d in deltas:
    dm = multi[d]["mean"]
    dlo = multi[d]["lo"]
    dhi = multi[d]["hi"]

    idx = np.argmin(dm)  # più negativo = massimo beneficio
    summary_rows.append({
        "delta": d,
        "best_at_severity": float(sev_grid[idx]),
        "deltaP_mean": float(dm[idx]),
        "deltaP_95lo": float(dlo[idx]),
        "deltaP_95hi": float(dhi[idx]),
    })

summary_rows

In [ ]:
for r in summary_rows:
    print(
        f"Δ={r['delta']}: best at s={r['best_at_severity']:.2f} | "
        f"ΔP={r['deltaP_mean']:.3f} [{r['deltaP_95lo']:.3f}, {r['deltaP_95hi']:.3f}]"
    )

In [ ]:
# Severity individuali
s_individual = df_subject["sev_z"].values
IDs = df_subject["ID"].values

print("N subjects:", len(s_individual))

In [ ]:
from scipy.stats import norm

def probit(x):
    return norm.cdf(x)

In [ ]:
# --- STEP 4 (robusto): build df_cf ---
import numpy as np
import pandas as pd
from scipy.stats import norm

def probit(x):
    return norm.cdf(x)

def individual_counterfactual(s, tau_samples, delta):
    p_before = probit(s - tau_samples)
    p_after  = probit((s - delta) - tau_samples)
    delta_p  = p_after - p_before
    return {
        "mean": float(delta_p.mean()),
        "hdi_2.5": float(np.quantile(delta_p, 0.025)),
        "hdi_97.5": float(np.quantile(delta_p, 0.975)),
    }

# hard checks
assert "df_subject" in globals(), "df_subject non esiste: esegui la cella che lo costruisce"
assert "tau_samples" in globals(), "tau_samples non esiste: esegui la cella che li estrae dal trace del threshold model"
assert "sev_z" in df_subject.columns, "df_subject deve contenere sev_z"
assert "target_bin" in df_subject.columns, "df_subject deve contenere target_bin"

s_individual = df_subject["sev_z"].to_numpy()
IDs = df_subject["ID"].to_numpy()
y_obs = df_subject["target_bin"].to_numpy().astype(int)

DELTA = 0.3

rows = []
for i, s in enumerate(s_individual):
    res = individual_counterfactual(s=s, tau_samples=tau_samples, delta=DELTA)
    rows.append({
        "ID": IDs[i],
        "sev_z": float(s),
        "delta": DELTA,
        "deltaP_mean": res["mean"],
        "deltaP_95lo": res["hdi_2.5"],
        "deltaP_95hi": res["hdi_97.5"],
        "observed_fall": int(y_obs[i]),
    })

df_cf = pd.DataFrame(rows)

print("df_cf built:", df_cf.shape)
display(df_cf.head())

In [ ]:
df_cf[["deltaP_mean"]].describe()

In [ ]:
plt.figure(figsize=(8,5))

plt.scatter(
    df_cf["sev_z"],
    df_cf["deltaP_mean"],
    c=df_cf["observed_fall"],
    cmap="coolwarm",
    alpha=0.8
)

plt.axhline(0, color="black", ls="--")
plt.xlabel("Latent motor severity (z)")
plt.ylabel(r"$\Delta P(\mathrm{fall})$ for $\Delta s=-0.3$")
plt.title("Individualized counterfactual fall-risk reduction")

plt.colorbar(label="Observed fall (0=no, 1=yes)")
plt.show()

In [ ]:
df_cf["clinically_relevant"] = df_cf["deltaP_mean"] < -0.05

print(
    "Subjects with ≥5% absolute risk reduction:",
    df_cf["clinically_relevant"].mean().round(3)
)

In [ ]:
# ===============================
# TOP-10 subjects by benefit
# ===============================

top10 = (
    df_cf
    .sort_values("deltaP_mean")  # più negativo = più beneficio
    .head(10)
    .loc[:, ["ID", "sev_z", "deltaP_mean", "deltaP_95lo", "deltaP_95hi", "observed_fall"]]
)

top10

In [ ]:
total_benefit = np.abs(df_cf["deltaP_mean"]).sum()

In [ ]:
df_sorted = df_cf.sort_values("sev_z").copy()

In [ ]:
df_sorted["benefit_abs"] = np.abs(df_sorted["deltaP_mean"])
df_sorted["benefit_cum"] = df_sorted["benefit_abs"].cumsum()
df_sorted["benefit_frac"] = df_sorted["benefit_cum"] / total_benefit

In [ ]:
zone_90 = df_sorted[df_sorted["benefit_frac"] <= 0.90]

sev_min = zone_90["sev_z"].min()
sev_max = zone_90["sev_z"].max()

sev_min, sev_max

In [ ]:
plt.figure(figsize=(7,4))
plt.plot(df_sorted["sev_z"], df_sorted["benefit_frac"], lw=2)
plt.axhline(0.9, color="red", ls="--", label="90% of total benefit")
plt.axvspan(sev_min, sev_max, color="red", alpha=0.15,
            label="Target severity zone")
plt.xlabel("Latent motor severity (z)")
plt.ylabel("Cumulative fraction of benefit")
plt.legend()
plt.title("Where counterfactual benefit concentrates")
plt.show()

In [ ]:
# -----------------------------
# Posterior samples
# -----------------------------
tau_samples = (
    trace_threshold.posterior["tau"]
    .stack(sample=("chain", "draw"))
    .values
)

true_sev_samples = (
    trace_threshold.posterior["true_severity"]
    .stack(sample=("chain", "draw"))
    .values
)

print("tau samples:", tau_samples.shape)
print("true severity samples:", true_sev_samples.shape)

In [ ]:
from scipy.stats import norm

n_samples = tau_samples.shape[0]
n_subjects = true_sev_samples.shape[0]

# P(fall | posterior)
p_fall_pp = norm.cdf(true_sev_samples[:, None] - tau_samples[None, :])

# Simulated falls
falls_sim = np.random.binomial(1, p_fall_pp)

print("Simulated falls matrix:", falls_sim.shape)

In [ ]:
# Observed prevalence
obs_prev = df_subject["target_bin"].mean()

# Simulated prevalence (per posterior sample)
sim_prev = falls_sim.mean(axis=0)

print("Observed prevalence:", obs_prev.round(3))
print("Simulated prevalence mean:", sim_prev.mean().round(3))
print("95% CI simulated prevalence:",
      np.quantile(sim_prev, [0.025, 0.975]).round(3))

In [ ]:
plt.figure(figsize=(8,5))

sns.histplot(sim_prev, bins=40, color="steelblue", alpha=0.7)
plt.axvline(obs_prev, color="red", lw=2, ls="--",
            label="Observed prevalence")

plt.xlabel("Fall prevalence")
plt.ylabel("Posterior predictive density")
plt.title("Posterior predictive check: fall prevalence")

plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
from scipy.stats import gaussian_kde

sev = df_cf["sev_z"].values
benefit = np.abs(df_cf["deltaP_mean"].values)

kde = gaussian_kde(sev, weights=benefit)
sev_grid = np.linspace(sev.min()-0.5, sev.max()+0.5, 300)
benefit_density = kde(sev_grid)

In [ ]:
# Normalizziamo la densità
benefit_density_norm = benefit_density / benefit_density.sum()

# Cumulativa
benefit_cum = np.cumsum(benefit_density_norm)

In [ ]:
low_target = sev_grid[np.where(benefit_cum >= 0.05)[0][0]]
high_target = sev_grid[np.where(benefit_cum >= 0.95)[0][0]]

print(f"Optimal intervention window: [{low_target:.2f}, {high_target:.2f}]")

In [ ]:
plt.figure(figsize=(8,4))

plt.plot(sev_grid, benefit_density, color="darkgreen", lw=3)

plt.fill_between(
    sev_grid,
    0,
    benefit_density,
    where=(sev_grid > low_target) & (sev_grid < high_target),
    color="green",
    alpha=0.3,
    label="Optimal intervention window (90% benefit)"
)

plt.axvline(0, color="gray", ls="--", lw=1)

plt.xlabel("Latent motor severity (z)")
plt.ylabel("Concentration of intervention benefit")

plt.title("Where fall-risk reduction concentrates")

plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
print([c for c in df_clin.columns if "Y" in c or "yahr" in c.lower() or "HY" in c])

In [ ]:
def clean_id(series):
    return (
        series
        .astype(str)
        .str.strip()
        .str.split()
        .str[0]
        .astype(int)
    )

In [ ]:
# Assicuriamoci che TUTTI i dataframe usino lo stesso ID
df_clin["ID"] = clean_id(df_clin["ID"])
df_subject["ID"] = clean_id(df_subject["ID"])

In [ ]:
HY_COL = "H-Y"

df_hy_subject = (
    df_clin
    .groupby("ID", as_index=False)
    .agg(HY=(HY_COL, "median"))
)

In [ ]:
print(df_hy_subject["ID"].dtype)
print(df_subject["ID"].dtype)

In [ ]:
df_subject = df_subject.merge(
    df_hy_subject,
    on="ID",
    how="left"
)

print(df_subject[["HY"]].describe())